In [0]:
# Cell 1 — read source tables
from pyspark.sql import functions as F

df_features = spark.table("raw.silver.cms_claims_features")
df_gold = spark.table("raw.gold.provider_risk")

print(f"Silver features rows: {df_features.count():,}")
print(f"Gold provider rows: {df_gold.count():,}")
print("\nGold schema preview:")
df_gold.select("provider_id", "specialty", "state", "risk_tier", "total_billed", "max_zscore").show(3, truncate=False)

Silver features rows: 1,999,992
Gold provider rows: 236,681

Gold schema preview:
+-----------+------------------+-----+---------+------------+----------+
|provider_id|specialty         |state|risk_tier|total_billed|max_zscore|
+-----------+------------------+-----+---------+------------+----------+
|1073992145 |Addiction Medicine|NY   |Critical |505057.0    |5.949     |
|1184724932 |Addiction Medicine|CA   |Critical |145985.0    |3.0648    |
|1134104300 |Addiction Medicine|RI   |Elevated |33000.0     |2.7666    |
+-----------+------------------+-----+---------+------------+----------+
only showing top 3 rows


In [0]:
# Cell 2 — KPI 1: Top 1% high cost claims
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_kpi1 = df_features \
    .withColumn("total_claim_value",
        F.col("avg_submitted_charge") * F.col("total_services")) \
    .withColumn("pct_rank",
        F.percent_rank().over(Window.orderBy("total_claim_value"))) \
    .filter(F.col("pct_rank") >= 0.99) \
    .select("provider_id", "specialty", "state", "procedure_code",
            "procedure_desc", "avg_submitted_charge", "avg_medicare_paid",
            "total_services", "total_beneficiaries", "total_claim_value")

print(f"KPI 1 — High cost claims (top 1%): {df_kpi1.count():,} records")
df_kpi1.orderBy(F.desc("total_claim_value")).show(3, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


KPI 1 — High cost claims (top 1%): 20,000 records
+-----------+----------------------------------------------+-----+--------------+-------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------+--------------+-------------------+-----------------+
|provider_id|specialty                                     |state|procedure_code|procedure_desc                                                                                                                                   |avg_submitted_charge|avg_medicare_paid|total_services|total_beneficiaries|total_claim_value|
+-----------+----------------------------------------------+-----+--------------+-------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------+--------------+-------------------+---------

In [0]:
# Cell 3 — KPI 2: Specialty benchmarks
df_kpi2 = df_features \
    .groupBy("specialty") \
    .agg(
        F.count("*").alias("total_claims"),
        F.countDistinct("provider_id").alias("unique_providers"),
        F.round(F.avg("avg_submitted_charge"), 2).alias("median_submitted_charge"),
        F.round(F.avg("charge_to_payment_ratio"), 4).alias("avg_charge_ratio"),
        F.round(F.avg("billing_intensity"), 4).alias("avg_billing_intensity"),
        F.round(F.sum(F.col("avg_submitted_charge") * F.col("total_services")), 2).alias("total_billed")
    ) \
    .orderBy(F.desc("total_billed"))

print(f"KPI 2 — Specialty benchmarks: {df_kpi2.count():,} specialties")
df_kpi2.show(5, truncate=False)

KPI 2 — Specialty benchmarks: 103 specialties
+--------------------------+------------+----------------+-----------------------+----------------+---------------------+---------------+
|specialty                 |total_claims|unique_providers|median_submitted_charge|avg_charge_ratio|avg_billing_intensity|total_billed   |
+--------------------------+------------+----------------+-----------------------+----------------+---------------------+---------------+
|Ambulatory Surgical Center|12269       |1054            |6572.03                |10.0836         |4.771                |5.3084662864E9 |
|Diagnostic Radiology      |229833      |6259            |308.59                 |6.9622          |2.1127               |4.28554771461E9|
|Internal Medicine         |176204      |18104           |219.98                 |5.177           |2.8156               |4.2550371132E9 |
|Ophthalmology             |45564       |3481            |495.41                 |4.2054          |2.3889               |3.997

In [0]:
# Cell 4 — KPI 3: State-level exposure
df_kpi3 = df_gold \
    .groupBy("state") \
    .agg(
        F.count("*").alias("total_providers"),
        F.sum("total_billed").alias("total_billed"),
        F.sum("total_paid").alias("total_paid"),
        F.round(F.avg("avg_charge_ratio"), 4).alias("avg_charge_ratio"),
        F.sum(F.when(F.col("risk_tier") == "Critical", 1).otherwise(0)).alias("critical_providers")
    ) \
    .orderBy(F.desc("total_billed"))

print(f"KPI 3 — State exposure: {df_kpi3.count():,} states")
df_kpi3.show(5, truncate=False)

# KPI 4: Procedure analysis
df_kpi4 = df_features \
    .groupBy("procedure_code", "procedure_desc") \
    .agg(
        F.count("*").alias("claim_count"),
        F.countDistinct("provider_id").alias("unique_providers"),
        F.round(F.avg("avg_submitted_charge"), 2).alias("avg_charge"),
        F.round(F.avg("avg_medicare_paid"), 2).alias("avg_paid"),
        F.round(F.avg("charge_to_payment_ratio"), 4).alias("avg_charge_ratio")
    ) \
    .orderBy(F.desc("claim_count"))

print(f"\nKPI 4 — Procedure analysis: {df_kpi4.count():,} procedures")
df_kpi4.show(5, truncate=False)

# KPI 5: Provider rankings within specialty
df_kpi5 = df_gold \
    .select("provider_id", "specialty", "state", "risk_tier",
            "specialty_rank", "total_billed", "max_zscore", "avg_charge_ratio") \
    .filter(F.col("risk_tier") == "Critical") \
    .orderBy("specialty", "specialty_rank")

print(f"\nKPI 5 — Critical provider rankings: {df_kpi5.count():,} providers")
df_kpi5.show(5, truncate=False)

# KPI 6: Patient concentration (potential mill providers)
df_kpi6 = df_gold \
    .filter(F.col("total_beneficiaries") > 0) \
    .withColumn("services_per_patient",
        F.round(F.col("total_services") / F.col("total_beneficiaries"), 2)) \
    .select("provider_id", "specialty", "state", "risk_tier",
            "total_beneficiaries", "total_services", "services_per_patient", "total_billed") \
    .orderBy(F.desc("services_per_patient"))

print(f"\nKPI 6 — Patient concentration: {df_kpi6.count():,} providers")
df_kpi6.show(5, truncate=False)

KPI 3 — State exposure: 60 states
+-----+---------------+--------------------+--------------------+----------------+------------------+
|state|total_providers|total_billed        |total_paid          |avg_charge_ratio|critical_providers|
+-----+---------------+--------------------+--------------------+----------------+------------------+
|CA   |18529          |8.017754482180031E9 |2.221875854999995E9 |6.2231          |1899              |
|FL   |15415          |6.097047558310006E9 |1.6874456421100016E9|5.4417          |1482              |
|TX   |15708          |5.971808279239997E9 |1.3668679905700023E9|6.5278          |1646              |
|NY   |16444          |5.0620072693400135E9|1.268707480759994E9 |6.0845          |1500              |
|NJ   |7197           |3.9819372421700034E9|9.019615195000021E8 |6.3572          |743               |
+-----+---------------+--------------------+--------------------+----------------+------------------+
only showing top 5 rows

KPI 4 — Procedure analy

In [0]:
# Cell 5 — write all KPI tables to Gold
df_kpi1.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw.gold.kpi_high_cost_claims")
print("KPI 1 written: high cost claims")

df_kpi2.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw.gold.kpi_specialty_benchmarks")
print("KPI 2 written: specialty benchmarks")

df_kpi3.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw.gold.kpi_state_exposure")
print("KPI 3 written: state exposure")

df_kpi4.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw.gold.kpi_procedure_analysis")
print("KPI 4 written: procedure analysis")

df_kpi5.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw.gold.kpi_critical_providers")
print("KPI 5 written: critical provider rankings")

df_kpi6.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("raw.gold.kpi_patient_concentration")
print("KPI 6 written: patient concentration")

print("\nAll Gold tables:")
spark.sql("SHOW TABLES IN raw.gold").show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


KPI 1 written: high cost claims
KPI 2 written: specialty benchmarks
KPI 3 written: state exposure
KPI 4 written: procedure analysis
KPI 5 written: critical provider rankings
KPI 6 written: patient concentration

All Gold tables:
+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|    gold|kpi_critical_prov...|      false|
|    gold|kpi_high_cost_claims|      false|
|    gold|kpi_patient_conce...|      false|
|    gold|kpi_procedure_ana...|      false|
|    gold|kpi_specialty_ben...|      false|
|    gold|  kpi_state_exposure|      false|
|    gold|       provider_risk|      false|
+--------+--------------------+-----------+

